In [1]:
!pip install -q openai pandas

import os
import time
from getpass import getpass

from openai import OpenAI

# Prefer Colab's secret manager if you've added OPENROUTER_API_KEY there
# (the key icon in the left sidebar). Falls back to a hidden prompt so the
# key is never typed into a visible cell or saved in the notebook file.
try:
    from google.colab import userdata
    api_key = userdata.get("OPENROUTER_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass("Enter your OpenRouter API key: ")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)


Enter your OpenRouter API key: ··········


In [8]:
MODELS = {
    "gpt-4o-mini":   "openai/gpt-4o-mini",
    "claude-haiku":  "anthropic/claude-haiku-4.5",   # current slug as of Oct 2026
    "gemini-flash":  "google/gemini-2.5-flash",      # current slug as of Oct 2026
    "llama-3.1":     "meta-llama/llama-3.1-70b-instruct",
}

In [3]:
def call_model(model_key: str, prompt: str, system: str = None,
                temperature: float = 0.7, max_tokens: int = 600,
                retries: int = 2) -> dict:
    """
    Send `prompt` to the model registered under `model_key` in MODELS,
    and return a dict with the response text plus metadata useful for
    comparison (latency, token usage, success/failure).

    Retries once on transient errors (timeouts, rate limits) before
    giving up and recording the failure, so one flaky call doesn't
    crash the whole comparison run.
    """
    if model_key not in MODELS:
        raise ValueError(f"Unknown model_key '{model_key}'. Options: {list(MODELS)}")

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    last_error = None
    for attempt in range(retries + 1):
        start = time.time()
        try:
            response = client.chat.completions.create(
                model=MODELS[model_key],
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            latency = round(time.time() - start, 2)
            choice = response.choices[0].message.content
            usage = getattr(response, "usage", None)

            return {
                "model_key": model_key,
                "model_slug": MODELS[model_key],
                "prompt": prompt,
                "output": choice,
                "latency_sec": latency,
                "prompt_tokens": getattr(usage, "prompt_tokens", None),
                "completion_tokens": getattr(usage, "completion_tokens", None),
                "total_tokens": getattr(usage, "total_tokens", None),
                "success": True,
                "error": None,
            }
        except Exception as e:
            last_error = str(e)
            time.sleep(1.5)  # brief backoff before retrying

    # All attempts failed — record the failure instead of crashing.
    return {
        "model_key": model_key,
        "model_slug": MODELS[model_key],
        "prompt": prompt,
        "output": None,
        "latency_sec": None,
        "prompt_tokens": None,
        "completion_tokens": None,
        "total_tokens": None,
        "success": False,
        "error": last_error,
    }

In [4]:
def run_comparison(prompt: str, model_keys: list, system: str = None,
                    **kwargs) -> list:
    """
    Send one prompt to each model in `model_keys` and collect results.
    Keeping the prompt identical across models is what makes the
    comparison meaningful — the model becomes the only variable.
    """
    results = []
    for key in model_keys:
        print(f"Calling {key} ...")
        result = call_model(key, prompt, system=system, **kwargs)
        results.append(result)
    return results

In [5]:
TASKS = {
    "creative_writing": {
        "system": "You are a skilled creative writer.",
        "prompt": (
            "Write a short poem (max 150 words) about a lighthouse keeper "
            "who has never seen the ocean."
        ),
    },
    "analytical_reasoning": {
        "system": "You are a precise analytical assistant.",
        "prompt": (
            "A train leaves City A at 60 mph heading toward City B, 300 miles "
            "away. At the same time, a train leaves City B at 40 mph heading "
            "toward City A. After how many hours do they meet, and how far "
            "from City A? Show your reasoning step by step, then give the "
            "final answer clearly labeled."
        ),
    },
    "instruction_following": {
        "system": "You follow formatting instructions exactly.",
        "prompt": (
            "Return ONLY valid JSON (no commentary, no markdown fences) "
            "representing a 4-item to-do list for 'launching a small bakery'. "
            "Each item must be an object with keys: 'task' (string), "
            "'priority' (one of 'high', 'medium', 'low'), and "
            "'estimated_days' (integer)."
        ),
    },
}

In [9]:
import pandas as pd

SELECTED_MODELS = ["gpt-4o-mini", "claude-haiku", "gemini-flash"]  # pick 2+

all_results = []
for task_name, task in TASKS.items():
    print(f"\n=== Task: {task_name} ===")
    task_results = run_comparison(
        prompt=task["prompt"],
        model_keys=SELECTED_MODELS,
        system=task["system"],
    )
    for r in task_results:
        r["task"] = task_name
    all_results.extend(task_results)

results_df = pd.DataFrame(all_results)
results_df = results_df[[
    "task", "model_key", "success", "latency_sec",
    "total_tokens", "output", "error"
]]
results_df  # displays the full comparison table in Colab


=== Task: creative_writing ===
Calling gpt-4o-mini ...
Calling claude-haiku ...
Calling gemini-flash ...

=== Task: analytical_reasoning ===
Calling gpt-4o-mini ...
Calling claude-haiku ...
Calling gemini-flash ...

=== Task: instruction_following ===
Calling gpt-4o-mini ...
Calling claude-haiku ...
Calling gemini-flash ...


,task,model_key,success,latency_sec,total_tokens,output,error
0,creative_writing,gpt-4o-mini,True,2.91,245,"In a tower tall, the keeper stands, \nWith la...",None
1,creative_writing,claude-haiku,True,3.28,234,# The Keeper's Paradox\n\nThirty years upon th...,None
2,creative_writing,gemini-flash,True,1.54,181,"Within a tower, stark and high,\nA keeper tend...",None
3,analytical_reasoning,gpt-4o-mini,True,3.76,474,To find out when and where the two trains meet...,None
4,analytical_reasoning,claude-haiku,True,3.03,454,# Train Meeting Problem - Step by Step Solutio...,None
5,analytical_reasoning,gemini-flash,True,2.11,443,Here's a step-by-step solution to determine wh...,None
6,instruction_following,gpt-4o-mini,True,1.85,203,"[\n {\n ""task"": ""Secure funding for ...",None
7,instruction_following,claude-haiku,True,1.49,248,"```json\n[\n {\n ""task"": ""Obtain business ...",None
8,instruction_following,gemini-flash,True,1.31,223,"[\n {\n ""task"": ""Finalize business p...",None


In [10]:
results_df.to_csv("multibot_comparison_results.csv", index=False)
print("Saved to multibot_comparison_results.csv")

# Print each output in full (CSV truncates long text) for easy copy/paste
for task_name in TASKS:
    print(f"\n{'='*60}\nTASK: {task_name}\n{'='*60}")
    for r in all_results:
        if r["task"] == task_name:
            print(f"\n--- {r['model_key']} (latency: {r['latency_sec']}s, "
                  f"tokens: {r['total_tokens']}, success: {r['success']}) ---")
            print(r["output"] if r["success"] else f"ERROR: {r['error']}")

Saved to multibot_comparison_results.csv

TASK: creative_writing

--- gpt-4o-mini (latency: 2.91s, tokens: 245, success: True) ---
In a tower tall, the keeper stands,  
With lantern bright, and gentle hands,  
He guards the shore, though never seen,  
The ocean’s dance, its restless sheen.  

Each night he lights the beacon's flame,  
A silent sentinel, without a name,  
The waves’ soft whispers, a distant song,  
Yet in his heart, he feels he belongs.  

He climbs the stairs, with steadfast grace,  
Imagining tides, the sea's embrace,  
He dreams of storms and ships that roam,  
While never tasting the salt of foam.  

From dawn to dusk, through fog and mist,  
He charts the skies, the stars persist,  
Though landlocked eyes may never roam,  
In guiding light, he finds his home.  

As seagulls cry and nightbirds call,  
The keeper stands, though he’s never seen it all,  
For in the light, he feels the ocean's sigh,  
A lighthouse heart, where dreams can fly.  

--- claude-haiku (laten